# 일단 '형식 검사' 먼저 구현

In [5]:
import re
import os
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import openpyxl
from openpyxl import load_workbook

In [9]:
# 샘플 파일 경로
settlement_file = "2025학년도_03월_음악과_학생회_결산안.xlsx"
bank_file = "2025학년도_03월_음악과_학생회_통장거래내역서_1차.xlsx"

# 실행 정보
audit_round = "1차"
org_name = "음악과 학생회"
target_month = "2025-03"
homepage_uploaded_at = "2025-04-05 22:30"   # 예시, 나중엔 실제 홈페이지 업로드 시각 입력

In [15]:
STANDARD_EVIDENCE_LABELS = {
    "입금증빙", "정상영수증", "카드전표", "거래명세표", "이체확인증",
    "전월이월금", "현금수령증", "단체현금수령증", "간이영수증",
    "상금수령증", "상품수령증", "고액상품수령증", "사유서",
    "전기이월금", "식대사용보고안", "추가증빙자료"
}


def safe_load_workbook(file_path, data_only=False):
    try:
        wb = load_workbook(file_path, data_only=data_only)
        return wb, None
    except Exception as e:
        return None, str(e)


def check_settlement_filename(file_name):
    pattern = r"^2025학년도[_ ]?\d{1,2}월[_ ].+[_ ]학생회[_ ]결산안[_ ](1차|2차|3차)\.xlsx$"
    passed = bool(re.match(pattern, file_name))
    return {
        "rule_id": "RULE-FMT-001",
        "rule_name": "결산안 파일명 형식",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else "결산안 파일명에 차수(1차/2차/3차)가 없거나 형식이 다릅니다.",
        "expected_fix": None if passed else "파일명을 '2025학년도_03월_음악과_학생회_결산안_1차.xlsx' 형식으로 수정하세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }


def check_bank_filename(file_name):
    pattern = r"^2025학년도[_ ]?\d{1,2}월[_ ].+[_ ]학생회[_ ]통장거래내역([_ ](1차|2차|3차))?\.xlsx$"
    passed = bool(re.match(pattern, file_name))
    return {
        "rule_id": "RULE-FMT-002",
        "rule_name": "통장거래내역 파일명 형식",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else "통장거래내역 파일명 형식이 교육자료 기준과 다릅니다.",
        "expected_fix": None if passed else "파일명을 '2025학년도_03월_음악과_학생회_통장거래내역_1차.xlsx' 형식으로 수정하세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }


def check_sheet_name(sheet_name, target_month):
    month_num = int(target_month.split("-")[1])

    valid_names = {
        f"{month_num}월 결산안",
        f"{month_num:02d}월 결산안"
    }

    passed = sheet_name in valid_names

    return {
        "rule_id": "RULE-FMT-003",
        "rule_name": "결산안 시트명",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else f"시트명이 허용 형식이 아닙니다. 현재: '{sheet_name}'",
        "expected_fix": None if passed else f"시트명을 '{month_num}월 결산안' 또는 '{month_num:02d}월 결산안'으로 수정하세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }


def check_sheet_protection(ws):
    protected = ws.protection.sheet
    return {
        "rule_id": "RULE-FMT-004",
        "rule_name": "결산안 시트 보호",
        "result_status": "PASS" if protected else "FAIL",
        "issue_summary": None if protected else "결산안 시트 보호가 적용되어 있지 않습니다.",
        "expected_fix": None if protected else "시트 보호를 적용하세요.",
        "predicted_penalty_points": 2 if not protected else 0
    }


def detect_header_row(ws):
    for row in ws.iter_rows(min_row=1, max_row=15):
        values = [cell.value for cell in row]
        if values and "날짜" in values and "거래처명" in values:
            return row[0].row
    return None


def get_column_index_map(ws, header_row):
    headers = [ws.cell(header_row, col).value for col in range(1, ws.max_column + 1)]
    return {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}


def check_formula_usage(ws):
    """
    결산안 잔액 수식 검사
    - 데이터 행마다 잔액 셀에 수식이 있어야 함
    - 첫 데이터 행: 같은 행의 수입/지출을 참조해야 함
    - 이후 행: 직전 행 잔액 + 현재 행 수입 - 현재 행 지출 구조여야 함
    """
    header_row = detect_header_row(ws)
    if header_row is None:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "헤더 행을 찾지 못해 수식 검사가 어렵습니다.",
            "expected_fix": "결산안 시트 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    col_map = get_column_index_map(ws, header_row)

    required_cols = ["날짜", "거래처명", "품목", "수입", "지출", "잔액"]
    missing = [c for c in required_cols if c not in col_map]
    if missing:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": f"필수 열을 찾지 못했습니다: {missing}",
            "expected_fix": "결산안 열 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    income_col_idx = col_map["수입"]
    expense_col_idx = col_map["지출"]
    balance_col_idx = col_map["잔액"]

    income_col_letter = get_column_letter(income_col_idx)
    expense_col_letter = get_column_letter(expense_col_idx)
    balance_col_letter = get_column_letter(balance_col_idx)

    def is_data_row(row_idx):
        values = {
            "날짜": ws.cell(row_idx, col_map["날짜"]).value,
            "거래처명": ws.cell(row_idx, col_map["거래처명"]).value,
            "품목": ws.cell(row_idx, col_map["품목"]).value,
            "수입": ws.cell(row_idx, income_col_idx).value,
            "지출": ws.cell(row_idx, expense_col_idx).value,
            "잔액": ws.cell(row_idx, balance_col_idx).value,
        }

        # 완전 공백 행은 제외
        return any(v is not None and str(v).strip() != "" for v in values.values())

    data_rows = [row_idx for row_idx in range(header_row + 1, ws.max_row + 1) if is_data_row(row_idx)]

    if not data_rows:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "데이터 행을 찾지 못했습니다.",
            "expected_fix": "결산안 데이터 영역을 확인하세요.",
            "predicted_penalty_points": 0
        }

    missing_formula_rows = []
    invalid_formula_rows = []

    for i, row_idx in enumerate(data_rows):
        balance_cell = ws.cell(row_idx, balance_col_idx)
        formula = balance_cell.value

        if not (isinstance(formula, str) and formula.startswith("=")):
            missing_formula_rows.append(row_idx)
            continue

        formula_upper = formula.upper().replace("$", "").replace(" ", "")

        # 현재 행의 수입/지출 참조 여부
        has_current_income = f"{income_col_letter}{row_idx}" in formula_upper
        has_current_expense = f"{expense_col_letter}{row_idx}" in formula_upper

        if i == 0:
            # 첫 데이터 행: 같은 행의 수입/지출을 참조하면 일단 통과
            if not (has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))
        else:
            prev_row_idx = data_rows[i - 1]
            has_prev_balance = f"{balance_col_letter}{prev_row_idx}" in formula_upper

            if not (has_prev_balance and has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))

    if missing_formula_rows or invalid_formula_rows:
        msgs = []
        if missing_formula_rows:
            if len(missing_formula_rows) <= 10:
                msgs.append(f"잔액 수식 누락 행: {missing_formula_rows}")
            else:
                msgs.append(f"잔액 수식 누락 행 다수 존재 (예: {missing_formula_rows[:10]} ...)")

        if invalid_formula_rows:
            sample = [f"{r}행({f})" for r, f in invalid_formula_rows[:5]]
            msgs.append(f"잔액 수식 구조 이상 행: {sample}")

        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "FAIL",
            "issue_summary": " / ".join(msgs),
            "expected_fix": "모든 데이터 행의 잔액 셀에 엑셀 수식을 적용하고, 첫 행 이후에는 직전 행 잔액을 참조하도록 통일하세요.",
            "predicted_penalty_points": 2
        }

    return {
        "rule_id": "RULE-FMT-005",
        "rule_name": "수식 사용 여부",
        "result_status": "PASS",
        "issue_summary": None,
        "expected_fix": None,
        "predicted_penalty_points": 0
    }


def extract_settlement_rows(ws):
    header_row = detect_header_row(ws)
    if header_row is None:
        return pd.DataFrame()

    col_map = get_column_index_map(ws, header_row)
    rows = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        date_val = ws.cell(row_idx, col_map.get("날짜", 1)).value if "날짜" in col_map else None
        vendor_val = ws.cell(row_idx, col_map.get("거래처명", 3)).value if "거래처명" in col_map else None
        item_val = ws.cell(row_idx, col_map.get("품목", 4)).value if "품목" in col_map else None

        if date_val is None and vendor_val is None and item_val is None:
            continue

        row_data = {"row_index": row_idx}
        for key, col_idx in col_map.items():
            row_data[key] = ws.cell(row_idx, col_idx).value
        rows.append(row_data)

    return pd.DataFrame(rows)


def check_evidence_label_standard(df):
    if df.empty or "증빙자료" not in df.columns:
        evidence_cols = [c for c in df.columns if "증빙" in str(c)]
        if not evidence_cols:
            return [{
                "rule_id": "RULE-FMT-008",
                "rule_name": "증빙자료명 표준화",
                "result_status": "REVIEW",
                "issue_summary": "증빙자료 관련 열을 찾지 못했습니다.",
                "expected_fix": "결산안 열 구조를 확인하세요.",
                "predicted_penalty_points": 0
            }]
        col_name = evidence_cols[0]
    else:
        col_name = "증빙자료"

    results = []

    for _, row in df.iterrows():
        value = row.get(col_name)
        if pd.isna(value) or value is None or str(value).strip() == "":
            continue

        labels = [x.strip() for x in str(value).split(",")]
        invalid_labels = [x for x in labels if x not in STANDARD_EVIDENCE_LABELS]

        if invalid_labels:
            results.append({
                "rule_id": "RULE-FMT-008",
                "rule_name": "증빙자료명 표준화",
                "result_status": "FAIL",
                "issue_summary": f"{int(row['row_index'])}행 증빙자료명 비표준 사용: {invalid_labels}",
                "expected_fix": "표준 증빙자료명으로 수정하세요.",
                "predicted_penalty_points": 2
            })

    if not results:
        results.append({
            "rule_id": "RULE-FMT-008",
            "rule_name": "증빙자료명 표준화",
            "result_status": "PASS",
            "issue_summary": None,
            "expected_fix": None,
            "predicted_penalty_points": 0
        })

    return results


def try_parse_datetime(val):
    if pd.isna(val):
        return None

    if isinstance(val, datetime):
        return val

    text = str(val).strip()
    patterns = [
        "%Y.%m.%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y.%m.%d %H:%M",
        "%Y-%m-%d %H:%M",
        "%Y.%m.%d",
        "%Y-%m-%d"
    ]
    for fmt in patterns:
        try:
            return datetime.strptime(text, fmt)
        except:
            pass
    return None

import re
import calendar
from openpyxl.utils import get_column_letter


def find_next_non_empty_in_row(ws, row_idx, start_col_idx):
    for col_idx in range(start_col_idx + 1, ws.max_column + 1):
        val = ws.cell(row_idx, col_idx).value
        if val is not None and str(val).strip() != "":
            return val
    return None


def extract_bank_query_period(ws):
    """
    통장 시트 상단에서 '조회기간' 값을 찾아 반환
    예: '2025.02.26 - 2025.03.31'
    """
    for row_idx in range(1, min(ws.max_row, 15) + 1):
        for col_idx in range(1, ws.max_column + 1):
            val = ws.cell(row_idx, col_idx).value
            if val is not None and str(val).strip() == "조회기간":
                period_val = find_next_non_empty_in_row(ws, row_idx, col_idx)
                if period_val is not None:
                    return str(period_val).strip()
    return None


def check_bank_query_period(ws, target_month):
    """
    통장거래내역 조회기간이 대상 월의 1일 ~ 말일인지 검사
    """
    period_text = extract_bank_query_period(ws)

    if not period_text:
        return {
            "rule_id": "RULE-FMT-009",
            "rule_name": "통장 조회기간",
            "result_status": "REVIEW",
            "issue_summary": "통장 시트에서 조회기간을 찾지 못했습니다.",
            "expected_fix": "통장거래내역 상단의 조회기간을 확인하세요.",
            "predicted_penalty_points": 0
        }

    m = re.search(r"(\d{4})[.\-](\d{2})[.\-](\d{2})\s*[-~]\s*(\d{4})[.\-](\d{2})[.\-](\d{2})", period_text)
    if not m:
        return {
            "rule_id": "RULE-FMT-009",
            "rule_name": "통장 조회기간",
            "result_status": "REVIEW",
            "issue_summary": f"조회기간 형식을 해석하지 못했습니다. 현재 값: {period_text}",
            "expected_fix": "조회기간 형식을 확인하세요.",
            "predicted_penalty_points": 0
        }

    start_y, start_m, start_d, end_y, end_m, end_d = map(int, m.groups())

    year, month = map(int, target_month.split("-"))
    last_day = calendar.monthrange(year, month)[1]

    expected_start = (year, month, 1)
    expected_end = (year, month, last_day)
    actual_start = (start_y, start_m, start_d)
    actual_end = (end_y, end_m, end_d)

    passed = (actual_start == expected_start) and (actual_end == expected_end)

    return {
        "rule_id": "RULE-FMT-009",
        "rule_name": "통장 조회기간",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else f"조회기간이 당월 1일~말일이 아닙니다. 현재: {period_text}",
        "expected_fix": None if passed else f"조회기간을 {year}.{month:02d}.01 - {year}.{month:02d}.{last_day:02d} 로 설정하세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }

def find_bank_header_row(ws):
    for row in ws.iter_rows(min_row=1, max_row=20):
        values = [cell.value for cell in row]
        values_str = [str(v).strip() if v is not None else "" for v in values]

        if "거래일시" in values_str and "구분" in values_str and "거래금액" in values_str:
            return row[0].row
    return None

def extract_bank_table_datetimes(ws):
    header_row = find_bank_header_row(ws)
    if header_row is None:
        return []

    headers = [ws.cell(header_row, col).value for col in range(1, ws.max_column + 1)]
    col_map = {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}

    if "거래일시" not in col_map:
        return []

    dt_col = col_map["거래일시"]
    dts = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        val = ws.cell(row_idx, dt_col).value
        dt = try_parse_datetime(val)
        if dt:
            dts.append(dt)

    return dts

def check_bank_row_order(ws):
    dts = extract_bank_table_datetimes(ws)

    if len(dts) < 2:
        return {
            "rule_id": "RULE-FMT-006",
            "rule_name": "통장 정렬 방향",
            "result_status": "REVIEW",
            "issue_summary": "거래 테이블의 거래일시를 충분히 찾지 못했습니다.",
            "expected_fix": "통장거래내역 형식을 확인하세요.",
            "predicted_penalty_points": 0
        }

    ascending = all(dts[i] <= dts[i + 1] for i in range(len(dts) - 1))

    return {
        "rule_id": "RULE-FMT-006",
        "rule_name": "통장 정렬 방향",
        "result_status": "PASS" if ascending else "FAIL",
        "issue_summary": None if ascending else "통장거래내역이 과거→최신 순으로 정렬되어 있지 않습니다.",
        "expected_fix": None if ascending else "과거 거래가 위로 오도록 저장하세요.",
        "predicted_penalty_points": 2 if not ascending else 0
    }

def check_deadline(homepage_uploaded_at, target_month):
    uploaded_dt = datetime.strptime(homepage_uploaded_at, "%Y-%m-%d %H:%M")
    year, month = map(int, target_month.split("-"))
    deadline = datetime(year, month + 1, 5, 23, 59) if month < 12 else datetime(year + 1, 1, 5, 23, 59)

    delta = uploaded_dt - deadline

    if uploaded_dt <= deadline:
        status = "PASS"
        summary = "기한 내 제출"
        penalty = 0
    else:
        late_days = delta.days + (1 if delta.seconds > 0 else 0)
        if 1 <= late_days <= 3:
            status = "FAIL"
            summary = f"{late_days}일 지각 제출"
            penalty = 5
        elif late_days >= 4:
            status = "FAIL"
            summary = f"{late_days}일 이상 지각 제출"
            penalty = 10
        else:
            status = "REVIEW"
            summary = "제출기한 판정 필요"
            penalty = 0

    return {
        "rule_id": "RULE-PRC-001",
        "rule_name": "제출기한 준수 여부",
        "result_status": status,
        "issue_summary": summary,
        "expected_fix": None if status == "PASS" else "홈페이지 업로드 기한을 준수해야 합니다.",
        "predicted_penalty_points": penalty
    }

In [16]:
results = []

settlement_name = Path(settlement_file).name
bank_name = Path(bank_file).name

results.append(check_settlement_filename(settlement_name))
results.append(check_bank_filename(bank_name))

settlement_wb, settlement_err = safe_load_workbook(settlement_file, data_only=False)
bank_wb, bank_err = safe_load_workbook(bank_file, data_only=False)

if settlement_wb:
    ws = settlement_wb[settlement_wb.sheetnames[0]]
    results.append(check_sheet_name(ws.title, target_month))
    results.append(check_sheet_protection(ws))
    results.append(check_formula_usage(ws))

    settlement_df = extract_settlement_rows(ws)
    results.extend(check_evidence_label_standard(settlement_df))
else:
    results.append({
        "rule_id": "SETTLEMENT-LOAD",
        "rule_name": "결산안 파일 열기",
        "result_status": "BLOCKED",
        "issue_summary": settlement_err,
        "expected_fix": "파일 상태를 확인하세요.",
        "predicted_penalty_points": 0
    })

if bank_wb:
    bank_ws = bank_wb[bank_wb.sheetnames[0]]
    results.append(check_bank_row_order(bank_ws))
    results.append(check_bank_query_period(bank_ws, target_month))
else:
    results.append({
        "rule_id": "BANK-LOAD",
        "rule_name": "통장 파일 열기",
        "result_status": "BLOCKED",
        "issue_summary": bank_err,
        "expected_fix": "암호 해제 또는 파일 상태 확인이 필요합니다.",
        "predicted_penalty_points": 2
    })

results.append(check_deadline(homepage_uploaded_at, target_month))

result_df = pd.DataFrame(results)
result_df

c:\Users\daeha\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,rule_id,rule_name,result_status,issue_summary,expected_fix,predicted_penalty_points
0,RULE-FMT-001,결산안 파일명 형식,FAIL,결산안 파일명에 차수(1차/2차/3차)가 없거나 형식이 다릅니다.,파일명을 '2025학년도_03월_음악과_학생회_결산안_1차.xlsx' 형식으로 수정...,2
1,RULE-FMT-002,통장거래내역 파일명 형식,FAIL,통장거래내역 파일명 형식이 교육자료 기준과 다릅니다.,파일명을 '2025학년도_03월_음악과_학생회_통장거래내역_1차.xlsx' 형식으로...,2
2,RULE-FMT-003,결산안 시트명,PASS,NaN,NaN,0
3,RULE-FMT-004,결산안 시트 보호,FAIL,결산안 시트 보호가 적용되어 있지 않습니다.,시트 보호를 적용하세요.,2
4,RULE-FMT-005,수식 사용 여부,FAIL,잔액 수식 누락 행: [4] / 잔액 수식 구조 이상 행: ['112행(=E112-...,"모든 데이터 행의 잔액 셀에 엑셀 수식을 적용하고, 첫 행 이후에는 직전 행 잔액을...",2
5,RULE-FMT-008,증빙자료명 표준화,PASS,NaN,NaN,0
6,RULE-FMT-006,통장 정렬 방향,PASS,NaN,NaN,0
7,RULE-FMT-009,통장 조회기간,FAIL,조회기간이 당월 1일~말일이 아닙니다. 현재: 2025.02.26 - 2025.03.31,조회기간을 2025.03.01 - 2025.03.31 로 설정하세요.,2
8,RULE-PRC-001,제출기한 준수 여부,PASS,기한 내 제출,NaN,0
